# 检测真实 iXBRL 标签

这个 notebook 用来查明 creditors / liabilities 抽取失败的真实原因。

它不会只看 notebook 里预设的标签字典，而是从 ZIP 内真实 accounts 文件中抽出所有 iXBRL facts，然后检查：

- 文件里到底有哪些 `fact_name`
- creditors / liabilities / payable / borrow / debt 相关标签是否存在
- 这些标签是不是 numeric fact
- 它们使用的是 `instant` context 还是 `duration` context
- 是否因为 `context_end_date == period_end` 这种规则被误过滤
- 是否只是标签字典没覆盖


In [ ]:
from pathlib import Path
import csv
import os
import re
import zipfile
from collections import Counter, defaultdict
from xml.etree import ElementTree as ET

import pandas as pd


def find_repo_root():
    env_root = os.environ.get("PROJECT_ROOT")
    starts = []
    if env_root:
        starts.append(Path(env_root))
    starts.append(Path.cwd())

    for start in starts:
        start = start.resolve()
        for p in [start, *start.parents]:
            if (p / ".git").exists() and (p / "00 Data Preparation + EDA").exists():
                return p
    raise FileNotFoundError(
        "Cannot find repository root. Start Jupyter from inside the cloned repo, "
        "or set PROJECT_ROOT to the repository root."
    )


def require_exists(path, label="path"):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")
    return path


def first_existing(paths, label):
    paths = [Path(path) for path in paths]
    for path in paths:
        if path.exists():
            return path
    tried = "\n".join(f"- {path}" for path in paths)
    raise FileNotFoundError(f"Cannot find {label}. Tried:\n{tried}")


PROJECT_ROOT = find_repo_root()
FINANCIAL_DIR = PROJECT_ROOT / "00 Data Preparation + EDA" / "Financial Data"
SMALL_TEST_DIR = FINANCIAL_DIR / "Small Scale Data Test"
SAMPLE_COMPANIES_DIR = SMALL_TEST_DIR / "Sample Companies"
LOCAL_DATA_ROOT = Path(os.environ.get(
    "LOCAL_DATA_ROOT",
    r"E:\000硕士毕设\财务数据\Local Large Data",
))
ACCOUNTS_ZIP_DIR = first_existing(
    [
        LOCAL_DATA_ROOT / "Accounts Data_2025.1_2026.5",
        FINANCIAL_DIR / "Accounts Data_2025.1_2026.5",
    ],
    "accounts ZIP directory",
)
BULK_MATCH_DIR = SMALL_TEST_DIR / "2025全年bulk匹配"
MODEL_PREP_DIR = BULK_MATCH_DIR / "模型训练与数据处理"
MODEL_TRAIN_DIR = MODEL_PREP_DIR / "模型训练"

require_exists(FINANCIAL_DIR, "financial data directory")
require_exists(SMALL_TEST_DIR, "small scale test directory")
require_exists(ACCOUNTS_ZIP_DIR, "accounts ZIP directory")


BASE_DIR = SMALL_TEST_DIR
ZIP_DIR = ACCOUNTS_ZIP_DIR

# 默认先诊断 sample。若想看 50k candidate 命中样本，改成 candidate_accounts_bulk_matches.csv。
MATCHES_FILE = BASE_DIR / "sample_accounts_bulk_matches.csv"

# 先跑小批量看标签；确认后可改为 None 跑全部 sample matches。
MAX_FILES = 300

OUTPUT_ALL_FACTS = BASE_DIR / "diagnostic_all_numeric_facts_long.csv"
OUTPUT_KEYWORD_FACTS = BASE_DIR / "diagnostic_creditor_liability_keyword_facts.csv"
OUTPUT_TAG_COUNTS = BASE_DIR / "diagnostic_fact_name_counts.csv"
OUTPUT_FAILURE_REPORT = BASE_DIR / "diagnostic_liability_failure_report.csv"

require_exists(MATCHES_FILE, "matches file")
if not sorted(ZIP_DIR.glob("Accounts_Monthly_Data-*.zip")):
    raise FileNotFoundError(f"No monthly accounts ZIP files found in {ZIP_DIR}")

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LOCAL_DATA_ROOT:", LOCAL_DATA_ROOT)
print("Matches:", MATCHES_FILE)
print("ZIP dir:", ZIP_DIR)


## 1. 读取匹配文件

每一行对应一个公司 accounts 文件：`source_zip + internal_filename`。


In [ ]:
matches = pd.read_csv(MATCHES_FILE, dtype=str)
matches["CompanyNumber"] = matches["CompanyNumber"].str.zfill(8)
matches = matches.drop_duplicates(subset=["source_zip", "internal_filename"]).reset_index(drop=True)

if MAX_FILES is not None:
    matches = matches.head(MAX_FILES).copy()

print("Files to inspect:", len(matches))
display(matches.head())
display(matches[["Accounts_AccountCategory", "account_category_group", "primary_sector"]].describe(include="all"))


## 2. 通用 iXBRL parser

注意这里抽取 **所有 numeric facts**，不是 turnover-only。这样可以判断 creditors/liabilities 是不是在之前的 turnover 过滤阶段就被丢掉了。


In [ ]:
def local_name(tag):
    return str(tag).rsplit("}", 1)[-1].split(":")[-1]


def strip_namespace(value):
    return str(value).split(":")[-1] if value else ""


def normalise_period_end(value):
    value = "" if pd.isna(value) else str(value)
    digits = re.sub(r"\D", "", value)
    if len(digits) == 8:
        return f"{digits[:4]}-{digits[4:6]}-{digits[6:8]}"
    return ""


def parse_numeric_value(text, scale=None, sign=None):
    if text is None:
        return None
    cleaned = re.sub(r"[\s,\u00a0£$€]", "", str(text))
    if cleaned in {"", "-", "—", "–"}:
        return None
    is_parenthesised_negative = cleaned.startswith("(") and cleaned.endswith(")")
    cleaned = cleaned.strip("()")
    try:
        value = float(cleaned)
    except ValueError:
        return None
    if scale not in (None, ""):
        try:
            value *= 10 ** int(scale)
        except ValueError:
            pass
    if str(sign).strip() == "-" or is_parenthesised_negative:
        value *= -1
    return value


def parse_contexts(root):
    contexts = {}
    for el in root.iter():
        if local_name(el.tag) != "context":
            continue
        context_id = el.attrib.get("id", "")
        if not context_id:
            continue
        info = {
            "context_ref": context_id,
            "context_period_type": "",
            "context_start_date": "",
            "context_end_date": "",
            "context_instant_date": "",
            "context_dimension_count": 0,
        }
        for child in el.iter():
            name = local_name(child.tag)
            text = "".join(child.itertext()).strip()
            if name == "startDate":
                info["context_start_date"] = text
                info["context_period_type"] = "duration"
            elif name == "endDate":
                info["context_end_date"] = text
                info["context_period_type"] = "duration"
            elif name == "instant":
                info["context_instant_date"] = text
                info["context_period_type"] = "instant"
            elif name in {"explicitMember", "typedMember"}:
                info["context_dimension_count"] += 1
        contexts[context_id] = info
    return contexts


def parse_ixbrl_facts(data):
    root = ET.fromstring(data)
    contexts = parse_contexts(root)
    facts = []
    for el in root.iter():
        tag = local_name(el.tag)
        if tag not in {"nonFraction", "nonNumeric"}:
            continue
        raw_name = el.attrib.get("name", "")
        fact_name = strip_namespace(raw_name)
        context_ref = el.attrib.get("contextRef", "")
        raw_value = "".join(el.itertext()).strip()
        numeric_value = parse_numeric_value(raw_value, el.attrib.get("scale"), el.attrib.get("sign"))
        context = contexts.get(context_ref, {})
        facts.append({
            "raw_name": raw_name,
            "fact_name": fact_name,
            "ix_tag_type": tag,
            "context_ref": context_ref,
            "unit_ref": el.attrib.get("unitRef", ""),
            "decimals": el.attrib.get("decimals", ""),
            "scale": el.attrib.get("scale", ""),
            "format": el.attrib.get("format", ""),
            "sign": el.attrib.get("sign", ""),
            "raw_value": raw_value,
            "numeric_value": numeric_value,
            **context,
        })
    return facts


## 3. 抽取所有 numeric facts

如果 creditors/liabilities 在这里存在，说明 parser 没问题，是后续标签映射或筛选规则的问题。


In [ ]:
all_numeric_rows = []
errors = []
zip_cache = {}

for i, row in matches.iterrows():
    if (i + 1) % 50 == 0:
        print(f"Processed {i + 1}/{len(matches)}")

    zip_name = row["source_zip"]
    internal_filename = row["internal_filename"]
    zip_path = ZIP_DIR / zip_name
    period_end_iso = normalise_period_end(row.get("period_end", ""))

    try:
        if zip_name not in zip_cache:
            zip_cache[zip_name] = zipfile.ZipFile(zip_path)
        data = zip_cache[zip_name].read(internal_filename)
        facts = parse_ixbrl_facts(data)

        for fact in facts:
            if fact["numeric_value"] is None:
                continue
            all_numeric_rows.append({
                "CompanyNumber": row.get("CompanyNumber", ""),
                "CompanyName": row.get("CompanyName", ""),
                "primary_sector": row.get("primary_sector", ""),
                "Accounts_AccountCategory": row.get("Accounts_AccountCategory", ""),
                "account_category_group": row.get("account_category_group", ""),
                "source_zip": zip_name,
                "internal_filename": internal_filename,
                "period_end": row.get("period_end", ""),
                "period_end_iso": period_end_iso,
                **fact,
            })
    except Exception as exc:
        errors.append({
            "CompanyNumber": row.get("CompanyNumber", ""),
            "CompanyName": row.get("CompanyName", ""),
            "source_zip": zip_name,
            "internal_filename": internal_filename,
            "error_type": type(exc).__name__,
            "error_message": str(exc)[:500],
        })

for z in zip_cache.values():
    z.close()

all_numeric = pd.DataFrame(all_numeric_rows)
errors_df = pd.DataFrame(errors)

print("Numeric facts:", len(all_numeric))
print("Files with errors:", len(errors_df))
display(all_numeric.head())


## 4. 关键词发现：真实标签到底叫什么

这里不使用预设标签，只用关键词搜索真实 `fact_name`。


In [ ]:
KEYWORDS = [
    "creditor", "liabil", "payable", "borrow", "loan", "debt",
    "asset", "cash", "bank", "debtor", "receivable",
]
keyword_pattern = "|".join(KEYWORDS)

keyword_facts = all_numeric[
    all_numeric["fact_name"].str.lower().str.contains(keyword_pattern, na=False)
].copy()

print("Keyword numeric facts:", len(keyword_facts))
display(keyword_facts[[
    "CompanyNumber", "CompanyName", "Accounts_AccountCategory", "fact_name",
    "raw_value", "numeric_value", "context_period_type", "context_instant_date",
    "context_end_date", "context_dimension_count", "unit_ref", "scale", "format"
]].head(100))

tag_counts = (
    all_numeric.groupby("fact_name", dropna=False)
    .agg(
        fact_rows=("fact_name", "size"),
        companies=("CompanyNumber", "nunique"),
        example_value=("numeric_value", "first"),
        context_types=("context_period_type", lambda s: ",".join(sorted(set(map(str, s))))),
    )
    .reset_index()
    .sort_values(["companies", "fact_rows"], ascending=False)
)

keyword_tag_counts = tag_counts[
    tag_counts["fact_name"].str.lower().str.contains(keyword_pattern, na=False)
].copy()

display(keyword_tag_counts.head(100))


## 5. 验证是不是 context 规则导致失败

资产负债表项目通常是 `instant` context，要用 `context_instant_date == period_end` 判断。  
P&L 项目如 turnover 通常是 `duration` context，才看 `context_end_date == period_end`。


In [ ]:
if not keyword_facts.empty:
    keyword_facts["instant_matches_period_end"] = keyword_facts["context_instant_date"].eq(keyword_facts["period_end_iso"])
    keyword_facts["duration_end_matches_period_end"] = keyword_facts["context_end_date"].eq(keyword_facts["period_end_iso"])

    context_report = (
        keyword_facts.groupby(["fact_name", "context_period_type"], dropna=False)
        .agg(
            rows=("fact_name", "size"),
            companies=("CompanyNumber", "nunique"),
            instant_match_rate=("instant_matches_period_end", "mean"),
            duration_end_match_rate=("duration_end_matches_period_end", "mean"),
            median_value=("numeric_value", "median"),
        )
        .reset_index()
        .sort_values(["companies", "rows"], ascending=False)
    )
else:
    context_report = pd.DataFrame()

display(context_report.head(100))


## 6. 用当前/旧标签字典做失败诊断

把你们 notebook 里尝试过的标签放在 `CURRENT_LIABILITY_TAGS` 中。这个单元会告诉你：

- 这些标签是否真的出现在文件里
- 如果没出现，真实相近标签是什么
- 如果出现了，是不是被 context 规则过滤掉了


In [ ]:
CURRENT_LIABILITY_TAGS = {
    "creditors_due_within_one_year": [
        "CreditorsDueWithinOneYear",
        "CreditorsAmountsFallingDueWithinOneYear",
        "CurrentLiabilities",
        "TotalCurrentLiabilities",
    ],
    "creditors_due_after_one_year": [
        "CreditorsDueAfterOneYear",
        "CreditorsAmountsFallingDueAfterMoreThanOneYear",
        "NoncurrentLiabilities",
    ],
    "net_assets": [
        "NetAssetsLiabilities",
        "NetAssets",
    ],
    "current_assets": [
        "CurrentAssets",
    ],
    "cash": [
        "CashBankOnHand",
        "CashAndCashEquivalents",
    ],
}

flat_expected_tags = sorted({tag for tags in CURRENT_LIABILITY_TAGS.values() for tag in tags})
observed_tags = set(all_numeric["fact_name"].dropna().astype(str))

diagnostic_rows = []
for standard_field, tags in CURRENT_LIABILITY_TAGS.items():
    for tag in tags:
        subset = all_numeric[all_numeric["fact_name"].eq(tag)].copy()
        if subset.empty:
            diagnostic_rows.append({
                "standard_field": standard_field,
                "expected_tag": tag,
                "observed_rows": 0,
                "observed_companies": 0,
                "instant_match_rate": None,
                "duration_end_match_rate": None,
                "diagnosis": "tag_not_observed_in_sample",
            })
        else:
            subset["instant_matches_period_end"] = subset["context_instant_date"].eq(subset["period_end_iso"])
            subset["duration_end_matches_period_end"] = subset["context_end_date"].eq(subset["period_end_iso"])
            likely_context_issue = (
                subset["instant_matches_period_end"].mean() > 0
                and subset["duration_end_matches_period_end"].mean() == 0
            )
            diagnostic_rows.append({
                "standard_field": standard_field,
                "expected_tag": tag,
                "observed_rows": len(subset),
                "observed_companies": subset["CompanyNumber"].nunique(),
                "instant_match_rate": subset["instant_matches_period_end"].mean(),
                "duration_end_match_rate": subset["duration_end_matches_period_end"].mean(),
                "diagnosis": "likely_context_filter_issue" if likely_context_issue else "tag_observed_check_selection_rule",
            })

failure_report = pd.DataFrame(diagnostic_rows)
display(failure_report)

print("Expected tags not observed:")
print([tag for tag in flat_expected_tags if tag not in observed_tags])


## 7. 自动生成候选标签建议

如果旧字典没命中，这里按真实标签频率给出候选。


In [ ]:
suggestions = {}
suggestion_patterns = {
    "creditors_due_within_one_year": "creditor|currentliabil|amountsfallingduewithin|withinoneyear|payable",
    "creditors_due_after_one_year": "creditor|noncurrentliabil|aftermorethan|afteroneyear|longterm|borrow",
    "current_assets": "currentassets|cash|debtor|receivable|inventory|stock",
    "net_assets": "netassets|totalassetslesscurrentliabilities|equity|shareholder",
    "cash": "cash|bank",
}

for field, pattern in suggestion_patterns.items():
    s = tag_counts[tag_counts["fact_name"].str.lower().str.contains(pattern, na=False)].head(30)
    suggestions[field] = s["fact_name"].tolist()
    print("\n", field)
    display(s[["fact_name", "fact_rows", "companies", "context_types", "example_value"]])


## 8. 写出诊断文件

这些 CSV 可以用来精确更新后续 extraction notebook 的标签映射。


In [ ]:
all_numeric.to_csv(OUTPUT_ALL_FACTS, index=False, encoding="utf-8-sig")
keyword_facts.to_csv(OUTPUT_KEYWORD_FACTS, index=False, encoding="utf-8-sig")
tag_counts.to_csv(OUTPUT_TAG_COUNTS, index=False, encoding="utf-8-sig")
failure_report.to_csv(OUTPUT_FAILURE_REPORT, index=False, encoding="utf-8-sig")

print("Wrote:")
print(OUTPUT_ALL_FACTS)
print(OUTPUT_KEYWORD_FACTS)
print(OUTPUT_TAG_COUNTS)
print(OUTPUT_FAILURE_REPORT)


## 如何读结论

- 如果 `keyword_facts` 里能看到 liabilities/creditors 标签：parser 没问题，失败原因是标签字典或 context 过滤。
- 如果 `failure_report` 里 `instant_match_rate > 0` 但 `duration_end_match_rate = 0`：说明你之前用 turnover 的 duration 规则抽 balance sheet，导致被误过滤。
- 如果真实标签存在但不在 `CURRENT_LIABILITY_TAGS`：补标签字典即可。
- 如果 `all_numeric` 里完全没有相关标签：该类 accounts 文件确实没有披露对应 numeric fact，应该降级为 unavailable，而不是继续猜标签。
